<a href="https://colab.research.google.com/github/mardyweb/atml-pa0/blob/main/notebooks/task1_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# to print nvidia driver's status table + to confirm that a gpu is attached to this session
!nvidia-smi

In [ ]:
from google.colab import userdata, drive
import os

USERNAME = "mardyweb"
REPO     = "atml-pa0"
TOKEN    = userdata.get('GITHUB_TOKEN')

# storing the url in an environment variable:
os.environ['GIT_URL'] = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

# clones the repo only if it is not already here (safe to re-run):
if not os.path.exists(f"/content/{REPO}"):
    !git clone $GIT_URL
%cd /content/$REPO

!git config user.email "maryamw17@outlook.com"
!git config user.name "Maryam"

# restoring the CIFAR-10 (170MB) and the ResNet-152 weights (230MB) that were cached to Drive
drive.mount('/content/drive')
!mkdir -p data /root/.cache/torch/hub/checkpoints
!cp -r /content/drive/MyDrive/atml_data/* data/ 2>/dev/null
!cp -r /content/drive/MyDrive/atml_cache/* /root/.cache/torch/hub/checkpoints/ 2>/dev/null
print("ready")

In [ ]:
from utils import set_seed, get_device, subset_loaders, save_results, save_fig

# should print cuda:
print(get_device())
# should print cifar-10-batches-py:
!ls data

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, time

# to ensure runs are reproducible:
set_seed(42)
device = get_device()

# subset_loaders loads CIFAR-10 (which has 60,000 images, 10 classes), resizes each to 224*224
# it then normalises the image with ImageNet's mean and std (ResNet's pretrained filters were trained on that size and pixel
# distribution) -> anything else would degarde the features badly (?)
# 5000 CIFAR images (500 per class) for training, 1000 for validation (100 per class)
# images are processed in batches of 64:
train_loader, val_loader = subset_loaders(n_train=5000, n_val=1000, batch_size=64)

model = torchvision.models.resnet152(weights="IMAGENET1K_V1")

# freeze the entire backbone (no gradients computed for these weights, weights are frozen)
for p in model.parameters():
    p.requires_grad = False

# model.fc is the final layer, currently 2048 -> 1000 (# of imagenet classes)
# we replace this with a 10-class CIFAR head
# requires_grad=True here:
model.fc = nn.Linear(model.fc.in_features, 10)

# all weights copied to GPU memory:
model = model.to(device)

# trainable params are 2048*10 (FC layer) + 10 biases = 20490/60M params
# (this is also why training from scratch is redundant)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")

In [ ]:
# cross entropy is the multi-class classification loss
criterion = nn.CrossEntropyLoss()

# optimizer only accesses the params in the final layer (non-frozen):
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

def run_epoch(model, loader, train=False):
    """Does one complete pass over `loader`. Returns (average loss, accuracy)."""
    # model.eval() is used even when training. The backbone is frozen, so we do not want BatchNorm updating its running statistics on CIFAR data. nn.Linear
    # -> behaves identically in train/eval mode, so the head still learns normally.
    model.eval()
    total_loss, correct, n = 0.0, 0, 0

    # x: [64, 3, 224, 224] (64 images, 3 colour channels, 224x224 pixels)
    # y: [64] (the correct class index (0-9) for each image)

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            #gradients are cleared so that each batch isnt polluted by the previous batch:
            optimizer.zero_grad()
            # forward pass -> [64, 10] raw scores:
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            # applying the update to weights:
            optimizer.step()
        else:
            # no_grad() skips gradient bookkeeping entirely: faster, uses less
            # memory, and guarantees evaluation can't accidentally change
            # anything:
            with torch.no_grad():
                out = model(x)
                loss = criterion(out, y)

        # criterion returns average loss over the batch in loss, so we multiply by batch size before summing:
        total_loss += loss.item() * y.size(0)
        # argmax(1) picks highest scoring class per image
        correct    += (out.argmax(1) == y).sum().item()
        n          += y.size(0)

    return total_loss / n, correct / n

In [ ]:
# 5 passes over 5000 training images
EPOCHS = 5
hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    # updates weights:
    tl, ta = run_epoch(model, train_loader, train=True)
    # only measures:
    vl, va = run_epoch(model, val_loader,  train=False)

    hist["train_loss"].append(tl); hist["train_acc"].append(ta)
    hist["val_loss"].append(vl);   hist["val_acc"].append(va)

    print(f"epoch {ep}/{EPOCHS}  "
          f"train loss {tl:.4f} acc {ta:.4f} | "
          f"val loss {vl:.4f} acc {va:.4f}  ({time.time()-t0:.0f}s)")

#writes the metrics and exact config to results/task1_baseline.json
save_results("task1_baseline", {
    "config": {"model": "resnet152", "frozen_backbone": True, "epochs": EPOCHS,
               "lr": 0.001, "momentum": 0.9, "optimizer": "SGD",
               "n_train": 5000, "n_val": 1000, "batch_size": 64,
               "trainable_params": trainable, "total_params": total},
    "history": hist
})

In [ ]:
import matplotlib.pyplot as plt
ep = range(1, EPOCHS + 1) #x-axis has the epoch number

# two plots side by side (loss curves, and accuracy)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(ep, hist["train_loss"], label="Training Loss")
ax[0].plot(ep, hist["val_loss"],   label="Validation Loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[0].set_title("Loss over epochs"); ax[0].legend()

ax[1].plot(ep, hist["train_acc"], label="Training Accuracy")
ax[1].plot(ep, hist["val_acc"],   label="Validation Accuracy")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy")
ax[1].set_title("Accuracy over epochs"); ax[1].legend()

plt.tight_layout()
save_fig(fig, "task1_baseline_curves")
plt.show()

In [ ]:
# task 1.2:

import types

def forward_no_skip(self, x):
    """
    Replacement for a Bottleneck block's forward pass, with the residual removed.

    Normally this is what a ResNet block computes:
        out = F(x)        <- the three conv layers
        out = out + x     <- skip connection
        return relu(out)

    We drop that addition, so the block becomes three plain stacked convolutions.
    """
    out = self.conv1(x);   out = self.bn1(out);  out = self.relu(out)
    out = self.conv2(out); out = self.bn2(out);  out = self.relu(out)
    out = self.conv3(out); out = self.bn3(out)
    # we are removing the out += input from the original
    return self.relu(out)

# same seed as 1.1, so the new head starts from identical weights:
set_seed(42)

# building a second model, so we can compare with the 1.1 baseline model
model_ns = torchvision.models.resnet152(weights="IMAGENET1K_V1")

# freezing the backbone (identical to 1.1):
for p in model_ns.parameters():
    p.requires_grad = False

# final layer replaced with 10-class head:
model_ns.fc = nn.Linear(model_ns.fc.in_features, 10)

# ResNet-152's layer4 has 3 Bottleneck blocks: 0, 1, 2. we disable the skip connections in blocks 1 and 2
# not block 0 because it has a `downsample` branch that reshapes the tensor, so its skip path isn't a
# plain identity and removing it would change the block's output dimensions (?)

DISABLED = [1, 2]

for i in DISABLED:
    # getting the block object:
    blk = model_ns.layer4[i]
    # MethodType attaches the function to this block object only, so `self`
    # resolves to that block. every other block in the network keeps its residual connection.
    blk.forward = types.MethodType(forward_no_skip, blk)

model_ns = model_ns.to(device)
print(f"skip connections disabled in layer4 blocks {DISABLED}")

In [ ]:
# same seed means the same 5000 images as the baseline are used. Any difference in results is attributable to the missing skip connections,
# not different data:

set_seed(42)

train_loader, val_loader = subset_loaders(n_train=5000, n_val=1000, batch_size=64)

criterion = nn.CrossEntropyLoss()
# rebinding `optimizer` to the new model's head:
optimizer = optim.SGD(model_ns.fc.parameters(), lr=0.001, momentum=0.9)

hist_ns = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for ep in range(1, EPOCHS + 1):
    t0 = time.time()
    tl, ta = run_epoch(model_ns, train_loader, train=True)
    vl, va = run_epoch(model_ns, val_loader,  train=False)
    hist_ns["train_loss"].append(tl); hist_ns["train_acc"].append(ta)
    hist_ns["val_loss"].append(vl);   hist_ns["val_acc"].append(va)
    print(f"epoch {ep}/{EPOCHS}  train loss {tl:.4f} acc {ta:.4f} | "
          f"val loss {vl:.4f} acc {va:.4f}  ({time.time()-t0:.0f}s)")

save_results("task1_noskip", {
    "config": {"disabled_blocks": DISABLED, "layer": "layer4",
               "epochs": EPOCHS, "lr": 0.001, "momentum": 0.9,
               "n_train": 5000, "n_val": 1000, "batch_size": 64},
    "history": hist_ns
})

In [ ]:
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

# left: training dynamics
ax[0].plot(ep, hist["train_loss"],    label="With Skip Connections")
ax[0].plot(ep, hist_ns["train_loss"], label="Without Skip Connections")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Training Loss")
ax[0].set_title("Training Loss Comparison"); ax[0].legend()

# right: generalisation
ax[1].plot(ep, hist["val_acc"],    label="With Skip Connections")
ax[1].plot(ep, hist_ns["val_acc"], label="Without Skip Connections")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Validation Accuracy")
ax[1].set_title("Validation Accuracy Comparison"); ax[1].legend()

plt.tight_layout()
save_fig(fig, "task1_skip_comparison")
plt.show()

print(f"final val acc — with skips: {hist['val_acc'][-1]:.4f} | "
      f"without: {hist_ns['val_acc'][-1]:.4f}")

In [ ]:
# committing + pushing to git:

!git config --global core.editor true
!git add .
!git commit -m "Task 1.2: residual connections"
!git pull --no-rebase --no-edit $GIT_URL main
!git push $GIT_URL